# Análise somática no Colab

**Mutect2 ou VCF do DRAGEN → Ensembl VEP → CIViC → AMP/ASCO/CAP → revisão especializada**

A prática começa com o par **WP043/WP044**, recortado para **JAK2** em **GRCh37**. A tarefa é rastrear o mesmo alelo da evidência nas leituras até a decisão de revisão. Use um runtime **CPU**.

| Modo | Entrada | Uso |
|---|---|---|
| `bams_aula` | BAMs do repositório | Chamada pareada e avaliação de filtros |
| `vcf_pronto` com caminho | VCF filtrado do DRAGEN ou do Mutect2 e FASTA compatível | Interpretação a partir de chamadas existentes |
| `vcf_pronto` com caminho vazio | VCF sintético incluído | Exercício rápido e contingência para downloads |

Use dados autorizados para o ambiente do Colab. As consultas ao VEP REST enviam montagem por endpoint e posição/REF/ALT sem IDs de amostra nem métricas. Para um VCF próprio, a chamada externa é desativada até você habilitá-la. Um JSON salvo pelo notebook permite repetir a interpretação localmente.

[Material original da aula](https://github.com/renatopuga/somatico) · [Guideline AMP/ASCO/CAP](https://pubmed.ncbi.nlm.nih.gov/27993330/)

## 1. Configuração

Para a prática completa, mantenha `bams_aula` e execute as células em ordem. O primeiro uso baixa o GATK (cerca de 700 MB) e a referência. Se você já tem os arquivos, preencha `GATK_JAR` e `FASTA`.

Para começar imediatamente pela interpretação, escolha `vcf_pronto`, mantenha `VCF_ENTRADA` vazio e selecione `recorte` nas duas fontes. O VCF de exemplo usa métricas sintéticas. O recorte inclui a anotação histórica do hotspot e o EID1 publicado na documentação do CIViC. A coluna `civic_status` informa o escopo da busca.

In [ ]:
# @title Parâmetros da prática
MODO = "bams_aula"  # @param ["bams_aula", "vcf_pronto"]
VCF_ENTRADA = ""  # @param {type:"string"}
CALLER_VCF = "DRAGEN"  # @param ["DRAGEN", "Mutect2"]
MONTAGEM_VCF = "GRCh37"  # @param ["GRCh37", "GRCh38"]
TUMOR_VCF = ""  # @param {type:"string"}
NORMAL_VCF = ""  # @param {type:"string"}
FASTA = ""  # @param {type:"string"}
GATK_JAR = ""  # @param {type:"string"}

VEP_FONTE = "rest"  # @param ["rest", "arquivo", "recorte"]
VEP_ARQUIVO = ""  # @param {type:"string"}
AUTORIZAR_VEP_EXTERNO_PARA_VCF_PROPRIO = False  # @param {type:"boolean"}
CIVIC_FONTE = "mensal"  # @param ["mensal", "recorte"]
CIVIC_RELEASE = "2026-11-01"  # @param {type:"string"}
CIVIC_PASTA = ""  # @param {type:"string"}
USAR_RECORTES_SE_OFFLINE = True  # @param {type:"boolean"}

In [ ]:
# @title Carregar as funções e criar uma pasta para esta execução
from datetime import datetime, timezone
import json
from pathlib import Path
import subprocess
import sys
import tempfile
from zoneinfo import ZoneInfo

from IPython.display import display
import pandas as pd

URL_REPO = "https://github.com/renatopuga/somatico.git"
COMMIT_RECURSOS = "da905d7a7aa237c277c0ff914eda7c7c7e3ac059"
LOCAL = Path.cwd()
NO_COLAB = Path("/content").is_dir()
RAIZ = Path("/content") if NO_COLAB else Path(tempfile.gettempdir())
if (LOCAL / "aula_somatica" / "recorte_aula.json").is_file():
    REPO = LOCAL
else:
    REPO = RAIZ / f"somatico-colab-{COMMIT_RECURSOS[:12]}"
    if not (REPO / ".git").is_dir():
        with tempfile.TemporaryDirectory(dir=RAIZ) as pasta:
            checkout = Path(pasta) / "somatico"
            subprocess.run(["git", "init", "-q", str(checkout)], check=True)
            git = ["git", "-C", str(checkout)]
            subprocess.run(
                [*git, "fetch", "--depth", "1", "--no-tags", URL_REPO, COMMIT_RECURSOS], check=True,
            )
            subprocess.run([*git, "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)
            checkout.rename(REPO)

sys.path.insert(0, str(REPO))
from aula_somatica import anotacao, chamada, revisao, variantes

COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
AULA = REPO / "aula_somatica"
RECURSOS = AULA / "recorte_aula.json"
RECORTE = json.loads(RECURSOS.read_text(encoding="utf-8"))
CACHE = RAIZ / "somatico-cache"
CACHE.mkdir(exist_ok=True)
SAIDA = Path(tempfile.mkdtemp(prefix="somatico-run-", dir=RAIZ))
print(f"Commit {COMMIT[:12]} | Saídas em {SAIDA}")

## 2. Chamada, filtros e medidas por alelo

O Mutect2 usa a referência hg19 do cromossomo 9 com o identificador `9`, o par de BAMs e o gnomAD recortado. A chamada fica limitada a `9:5021937-5126899`. O notebook registra quantos sítios de referência têm cobertura para estimar contaminação. Quando disponível, a estimativa é exploratória por vir de um único recorte.

Para **VCF próprio**, selecione caller e montagem, informe o ID tumoral e o normal se houver, e use o FASTA correspondente ao VCF. O FASTA deve ter os mesmos nomes de contig do VCF. A coluna `original_record` documenta o registro antes da divisão ou do alinhamento à esquerda. O arquivo de trabalho conserva as métricas da aula. A tabela mostra o `FILTER` original e, quando houver, o filtro por alelo do Mutect2. Só chamadas aprovadas nos dois campos entram na anotação e na revisão.

[Filtragem do Mutect2](https://gatk.broadinstitute.org/hc/en-us/articles/30332021620379-FilterMutectCalls)

In [ ]:
# @title Gerar ou carregar o VCF e conferir as chamadas
if MODO not in {"bams_aula", "vcf_pronto"}:
    raise ValueError("Selecione bams_aula ou vcf_pronto.")
if MODO == "bams_aula" and VCF_ENTRADA.strip():
    raise ValueError("Para usar VCF_ENTRADA, selecione o modo vcf_pronto.")
VCF_PROPRIO = MODO == "vcf_pronto" and bool(VCF_ENTRADA.strip())
SINTETICO = MODO == "vcf_pronto" and not VCF_PROPRIO
if VCF_PROPRIO and VEP_FONTE == "rest" and not AUTORIZAR_VEP_EXTERNO_PARA_VCF_PROPRIO:
    raise ValueError(
        "Para o VCF próprio, escolha VEP_FONTE='arquivo' ou habilite explicitamente "
        "AUTORIZAR_VEP_EXTERNO_PARA_VCF_PROPRIO."
    )

ferramentas, metadados_chamada = {}, {}
referencia = Path(FASTA.strip()).expanduser() if FASTA.strip() else None
if MODO == "bams_aula":
    ferramentas = chamada.preparar_ferramentas(chamar_mutect2=True)
    vcf, referencia, tumor, normal, metadados_chamada = chamada.mutect2_aula(
        REPO, CACHE, SAIDA, referencia=referencia, jar=GATK_JAR.strip() or None,
    )
    montagem, caller = "GRCh37", "Mutect2"
elif VCF_PROPRIO:
    vcf = Path(VCF_ENTRADA.strip()).expanduser()
    montagem, caller = variantes.montagem_canonica(MONTAGEM_VCF), CALLER_VCF
    tumor, normal = TUMOR_VCF.strip(), NORMAL_VCF.strip() or None
    if not vcf.is_file() or referencia is None or not referencia.is_file():
        raise ValueError("Informe os caminhos locais de VCF_ENTRADA e FASTA.")
    variantes.verificar_vcf(vcf, montagem, tumor, normal)
    ferramentas = chamada.preparar_ferramentas(chamar_mutect2=False)
else:
    vcf = AULA / "exemplo_sintetico.vcf"
    montagem, caller = "GRCh37", "Exemplo sintético"
    tumor, normal, referencia = "TUMOR_DEMO", "NORMAL_DEMO", None

normalizado = vcf if SINTETICO else variantes.normalizar(
    vcf, referencia, montagem, tumor, normal, SAIDA / "normalizado.vcf",
)
chamadas, contagens, fontes_vcf = variantes.ler_chamadas(normalizado, montagem, tumor, normal)
passantes = [alelo for alelo, dados in chamadas.values() if dados["passa_qc"]]
print(f"{caller} | {montagem} | Tumor {tumor} | Normal {normal or 'ausente'}")
print(contagens)
if not passantes:
    print("Nenhuma chamada PASS no escopo de SNVs/indels. As tabelas de evidência e revisão sairão vazias.")
medidas = pd.DataFrame([dados for _, dados in chamadas.values()])
if not medidas.empty:
    display(medidas.drop(columns=["original_record"]).head(12))

## 3. Consequência molecular e evidência

O VEP mantém as consequências por transcrito. O cruzamento CIViC procura primeiro o alelo na mesma montagem, após harmonizar a representação de SNVs e indels. Um HGVS codificante com transcrito versionado pode indicar uma correspondência a revisar. Perfis compostos, sinais de curadoria e correspondências ambíguas ficam explícitos. A pertinência clínica depende da doença, finalidade e direção da evidência.

Para trabalhar com os TSVs já baixados, mantenha `CIVIC_FONTE="mensal"` e aponte `CIVIC_PASTA` para os quatro arquivos da mesma release. As versões de evidência e asserções são as de registros aceitos. Para retomar o VEP com dados já consultados, selecione `VEP_FONTE="arquivo"` e informe `vep_response.json`.

[VEP GRCh37](https://grch37.rest.ensembl.org/documentation/info/vep_region_post) · [Releases CIViC](https://civicdb.org/releases) · [Modelo de asserções CIViC](https://docs.civicdb.org/en/latest/curating/assertions.html)

In [ ]:
# @title Anotar as chamadas PASS com VEP
fallback_aula = USAR_RECORTES_SE_OFFLINE and not VCF_PROPRIO
pacote_vep = anotacao.anotar_vep(
    passantes, montagem, VEP_FONTE, RECORTE["vep"],
    arquivo=VEP_ARQUIVO.strip() or None, fallback=fallback_aula,
)
arquivo_vep = SAIDA / "vep_response.json"
arquivo_vep.write_text(json.dumps(pacote_vep, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
for aviso in pacote_vep["avisos"]:
    print(aviso)
print(f"Alelos PASS com anotação disponível {len(pacote_vep['registros'])}/{len(passantes)}")

In [ ]:
# @title Ligar a variante às evidências e asserções do CIViC
if passantes:
    tabelas_civic, metadados_civic = anotacao.obter_civic(
        CIVIC_FONTE, CIVIC_RELEASE, CACHE, RECORTE["civic"],
        pasta=CIVIC_PASTA.strip() or None, fallback=fallback_aula,
    )
else:
    tabelas_civic = {nome: [] for nome in anotacao.CIVIC_TABELAS}
    metadados_civic = {"escopo": "sem_chamadas_pass"}
linhas_variantes, linhas_civic = anotacao.reunir_anotacoes(
    chamadas, pacote_vep, tabelas_civic, metadados_civic["escopo"],
)
revisao.escrever_tsv(SAIDA / "variants.tsv", revisao.VARIANTES_COLUNAS, linhas_variantes)
revisao.escrever_tsv(SAIDA / "civic_evidence.tsv", anotacao.CIVIC_COLUNAS, linhas_civic)
print(f"CIViC | {metadados_civic['escopo']} | {metadados_civic.get('release', '')}")
if metadados_civic.get("aviso"):
    print(metadados_civic["aviso"])
display(pd.DataFrame(linhas_variantes, columns=revisao.VARIANTES_COLUNAS).head(12))
campos_civic = ["variant_key", "registro", "registro_id", "doenca", "finalidade", "direcao",
                "significancia", "nivel_evidencia_civic", "amp_category_civic", "comparacao_pv_simulada", "registro_url"]
display(pd.DataFrame(linhas_civic, columns=anotacao.CIVIC_COLUNAS)[campos_civic].head(15))

## 4. AMP/ASCO/CAP e revisão especializada

| Tier | Significado clínico | Nível AMP/ASCO/CAP |
|---|---|---|
| I | Forte | A ou B |
| II | Potencial | C ou D |
| III | Desconhecido | Em branco |
| IV | Benigno ou provavelmente benigno | Em branco |

O campo `nivel_evidencia_civic` descreve o tipo de estudo do item curado. O campo `amp_category_civic` informa o tier que a asserção traz para seu próprio contexto. Ambos orientam a consulta às fontes. A proposta e a decisão da revisão são preenchidas pelos participantes. Para Tier III, registre as fontes consultadas e o que sustenta a incerteza. Para Tier IV, cite a evidência de benignidade. Se faltar contexto, deixe a ficha pendente.

1. No cenário **observado**, confira DP, AD, VAF e filtro nos dois BAMs via VCF. Registre quais dados clínicos seriam necessários para interpretar a variante.
2. No cenário **PV simulada**, compare o EID1 com o ICC. Avalie em qual finalidade o ICC sustenta uma proposta de tier e quais critérios adicionais seriam necessários para o diagnóstico.
3. Troque a ficha com a outra dupla. Cada revisor registra sua decisão e dá a referência que sustenta a revisão. Mantenha explícita a pendência quando faltarem dados.

[AMP/ASCO/CAP, Li et al.](https://pubmed.ncbi.nlm.nih.gov/27993330/) · [ICC, critérios para policitemia vera](https://pmc.ncbi.nlm.nih.gov/articles/PMC9479031/) · [CIViC EID1](https://civicdb.org/evidence/1)

In [ ]:
# @title Mostrar as fichas ainda pendentes
fichas = revisao.ficha_revisao(chamadas)
display(pd.DataFrame(fichas, columns=revisao.REVISAO_COLUNAS))

In [ ]:
# Preencha uma entrada por combinação de variant_key e cenario. Mantenha a lista
# cumulativa ao voltar à célula. Use I/II/III/IV e A/B/C/D em maiúsculas.
REVISOES = [
    # {
    #     "variant_key": "GRCh37:9:5073770:G:T",
    #     "cenario": "pv_simulada",
    #     "tier_proposto": "",
    #     "nivel_proposto": "",
    #     "fonte_proposta": revisao.ICC_PV,
    #     "justificativa_proposta": "",
    #     "tier_revisado": "",
    #     "nivel_revisado": "",
    #     "fonte_revisada": "",
    #     "justificativa_revisao": "",
    #     "revisor": "",
    #     "pendencias": "",
    # },
]

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date().isoformat()
fichas_exportadas = revisao.com_revisoes(fichas, REVISOES, hoje=hoje)
revisao.escrever_tsv(SAIDA / "review.tsv", revisao.REVISAO_COLUNAS, fichas_exportadas)
display(pd.DataFrame(fichas_exportadas, columns=revisao.REVISAO_COLUNAS))

## 5. Exportar e conferir a trilha da variante

O ZIP reúne `variants.tsv`, `civic_evidence.tsv`, `review.tsv`, `run_manifest.json` e `vep_response.json`. O manifesto registra versões, hashes, fontes e o modo de entrada. Antes de concluir, localize o alelo de JAK2, sua evidência em outro contexto e uma chamada sem correspondência na fonte CIViC usada.

In [ ]:
# @title Gerar ZIP para a aula
from collections import Counter
from zipfile import ZIP_DEFLATED, ZipFile

BAIXAR_ZIP = False  # @param {type:"boolean"}
arquivos = ["variants.tsv", "civic_evidence.tsv", "review.tsv", "vep_response.json"]
manifesto = {
    "gerado_em": datetime.now(timezone.utc).isoformat(),
    "repo": URL_REPO,
    "commit": COMMIT,
    "modo": MODO,
    "origem": "sintetico" if SINTETICO else "vcf_fornecido" if VCF_PROPRIO else "bams_aula",
    "caller": caller,
    "fontes_header_vcf": fontes_vcf,
    "assembly": montagem,
    "amostras": {"tumor": tumor, "normal": normal},
    "vcf_sha256": chamada.checksum(vcf),
    "normalizado_sha256": chamada.checksum(normalizado),
    "fasta_sha256": chamada.checksum(referencia) if referencia else None,
    "recorte_sha256": chamada.checksum(RECURSOS),
    "ferramentas": ferramentas,
    "chamada": metadados_chamada,
    "contagens_vcf_normalizado": contagens,
    "vep": {k: v for k, v in pacote_vep.items() if k != "registros"},
    "vep_fontes_por_alelo": dict(Counter(r["fonte"] for r in pacote_vep["registros"].values())),
    "civic": metadados_civic,
    "sha256_saidas": {nome: chamada.checksum(SAIDA / nome) for nome in arquivos},
}
(SAIDA / "run_manifest.json").write_text(
    json.dumps(manifesto, indent=2, ensure_ascii=False) + "\n", encoding="utf-8",
)
zip_saida = SAIDA / "interpretacao_somatica.zip"
with ZipFile(zip_saida, "w", compression=ZIP_DEFLATED) as z:
    for nome in [*arquivos, "run_manifest.json"]:
        z.write(SAIDA / nome, arcname=nome)
print(zip_saida)
if NO_COLAB and BAIXAR_ZIP:
    from google.colab import files
    files.download(str(zip_saida))